In [ ]:
# Databricks notebook source



# RAG Complaint Summarization

This notebook uses the pre-built RAG pipeline to generate summaries for complaints.

Prerequisites: Run the "RAG Pipeline Setup" notebook first to build and persist the pipeline.



## Input: Reference Number



In [ ]:

# Create widget for reference number input
dbutils.widgets.text("reference_number", "CDCR-23502631", "Reference Number")

# Get the reference number
REFERENCE_NUMBER = dbutils.widgets.get("reference_number")

print(f"Generating summary for: {REFERENCE_NUMBER}")



## Configuration



In [ ]:

import pandas as pd
import numpy as np
import json
import requests
from typing import List, Tuple, Dict, Optional

# Configuration
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL = "https://adb-7941446833400015.15.azuredatabricks.net"

# LLM Configuration - Claude Sonnet 4.5 Conservative
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"
LLM_TEMPERATURE = 0.3
LLM_MAX_TOKENS = 5000

# Delta table locations
CATALOG = "cntrl-busops-dev"
SCHEMA = "complaints-1kh-gld"
CHUNKS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`complaint_chunks`"
DOC_LOOKUP_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`complaint_documents`"
METADATA_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`rag_pipeline_metadata`"

# Vector Search
VECTOR_SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.complaints_chunk_index"

# RAG Parameters
K_NEIGHBORS = 4

print("Configuration loaded")



## Load Pipeline Metadata



In [ ]:

# Verify pipeline has been set up
try:
    metadata_df = spark.sql(f"SELECT * FROM {METADATA_TABLE}").toPandas()
    
    if len(metadata_df) > 0:
        metadata = metadata_df.iloc[0].to_dict()
        print("Pipeline metadata loaded:")
        print(f"  Pipeline run date: {metadata['pipeline_run_date']}")
        print(f"  Number of complaints: {metadata['num_complaints']}")
        print(f"  Number of chunks: {metadata['num_chunks']}")
        print(f"  Embedding dimension: {metadata['embedding_dimension']}")
    else:
        raise Exception("Metadata table is empty")
except Exception as e:
    print("ERROR: RAG pipeline not found!")
    print(f"Error: {e}")
    print()
    print("Please run the 'RAG Pipeline Setup' notebook first to set up the pipeline.")
    dbutils.notebook.exit("Pipeline not found")



## Load Document Lookup



In [ ]:

# Load document lookup for retrieving full complaint text
doc_lookup = spark.sql(f"SELECT * FROM {DOC_LOOKUP_TABLE}").toPandas()
doc_dict = dict(zip(doc_lookup['reference_number'], doc_lookup['full_document']))

print(f"Loaded {len(doc_dict)} complaint documents")



## Embedding Function



In [ ]:

def _invocations_url(workspace_url: str, endpoint_name: str) -> str:
    return f"{workspace_url.rstrip('/')}/serving-endpoints/{endpoint_name}/invocations"


def _make_headers(token: str) -> dict:
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def _parse_embeddings(resp_json):
    """Handle common embedding response shapes."""
    if isinstance(resp_json, dict):
        if "embeddings" in resp_json and isinstance(resp_json["embeddings"], list):
            return resp_json["embeddings"]
        if "data" in resp_json:
            data = resp_json["data"]
            if isinstance(data, list) and data and isinstance(data[0], dict) and "embedding" in data[0]:
                return [row["embedding"] for row in data]
            if isinstance(data, dict) and "embeddings" in data:
                return data["embeddings"]
    raise ValueError(f"Unrecognized embeddings response shape")


def embed_query(
    text: str,
    workspace_url: str = WORKSPACE_URL,
    endpoint_name: str = metadata['embedding_endpoint'],
    token: str = DATABRICKS_TOKEN
) -> List[float]:
    """Embed a single query text."""
    url = _invocations_url(workspace_url, endpoint_name)
    headers = _make_headers(token)
    
    payload = {"input": [text]}
    response = requests.post(url, headers=headers, json=payload, timeout=60)
    
    if response.ok:
        vecs = _parse_embeddings(response.json())
        # L2 normalize
        vec = np.array(vecs[0], dtype="float32")
        norm = np.linalg.norm(vec)
        if norm > 1e-12:
            vec = vec / norm
        return vec.tolist()
    else:
        raise RuntimeError(f"Embedding call failed: {response.status_code} {response.text[:500]}")



## Vector Search



In [ ]:

def retrieve_similar_chunks(
    reference_number: str,
    k: int = K_NEIGHBORS,
    exclude_source: bool = True
) -> List[Tuple[str, float, str]]:
    """Retrieve similar chunks using Vector Search."""
    
    # Get a chunk from the source complaint to use as query
    source_chunks = spark.sql(f"""
        SELECT chunk_text, embedding 
        FROM {CHUNKS_TABLE} 
        WHERE reference_number = '{reference_number}'
        LIMIT 1
    """).toPandas()
    
    if len(source_chunks) == 0:
        return []
    
    query_embedding = source_chunks.iloc[0]['embedding']
    
    # Call Vector Search API
    url = f"{WORKSPACE_URL}/api/2.0/vector-search/indexes/{VECTOR_SEARCH_INDEX}/query"
    
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "query_vector": query_embedding,
        "num_results": k * 5,  # Get more to filter
        "columns": ["chunk_id", "reference_number", "section", "chunk_text"]
    }
    
    response = requests.post(url, headers=headers, json=payload, timeout=120)
    
    if not response.ok:
        raise RuntimeError(f"Vector Search failed: {response.status_code} {response.text[:500]}")
    
    results_raw = response.json()
    
    # Process results
    seen_refs = set([reference_number]) if exclude_source else set()
    results = []
    
    for row in results_raw.get("result", {}).get("data_array", []):
        chunk_id, ref_num, section, chunk_text, score = row
        
        # Skip source complaint
        if exclude_source and ref_num == reference_number:
            continue
        
        # Get full document for unique complaints
        if ref_num not in seen_refs:
            seen_refs.add(ref_num)
            full_doc = doc_dict.get(ref_num, chunk_text)
            results.append((ref_num, float(score), full_doc))
        
        if len(results) >= k:
            break
    
    return results



## LLM Integration



In [ ]:

def call_llm_endpoint(
    prompt: str, 
    endpoint_name: str = LLM_ENDPOINT,
    max_new_tokens: int = LLM_MAX_TOKENS, 
    temperature: float = LLM_TEMPERATURE
) -> str:
    """Send prompt to LLM endpoint."""
    
    url = f"{WORKSPACE_URL}/serving-endpoints/{endpoint_name}/invocations"
    
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "You are an expert complaints analyst."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_new_tokens,
        "temperature": temperature
    }
    
    response = requests.post(url, headers=headers, json=payload, timeout=120)
    
    if response.ok:
        result = response.json()
        return result["choices"][0]["message"]["content"]
    else:
        raise RuntimeError(f"LLM call failed: {response.status_code} {response.text[:500]}")


SUMMARY_PROMPT = """You are an expert complaints analyst writing a clear, factual and chronological summary of a customer complaint.

Use ONLY the provided context - do not add information that isn't present.

Your goal is to create a concise narrative (5-10 sentences) that describes the complaint journey in order of events.
Include:
- When and how the complaint was received
- What the customer raised or alleged
- How the bank investigated and communicated during the process
- Key findings, decisions, or redress outcomes
- Any SLA breaches, delays, or escalation to FOS if applicable
- When and how the complaint was closed

Write it in professional plain English, past tense, and in chronological order (oldest events first).
Avoid bullet points or headings - return one coherent paragraph.

---
TARGET COMPLAINT CONTEXT:
{target}

---
SIMILAR CASES (for style reference only, do not copy facts):
{neighbors}

Now write the chronological summary of the target complaint only.
"""


def generate_summary(reference_number: str, k_neighbors: int = K_NEIGHBORS) -> str:
    """Generate a RAG-enhanced summary."""
    
    if reference_number not in doc_dict:
        return f"Error: Complaint {reference_number} not found in pipeline."
    
    target_doc = doc_dict[reference_number]
    
    # Retrieve similar complaints
    similar = retrieve_similar_chunks(reference_number, k=k_neighbors)
    
    # Build context from similar cases
    neigh_text = "\n\n---\n\n".join([
        f"# Similar {i+1} ({ref}) (score: {score:.3f})\n{doc[:6000]}"
        for i, (ref, score, doc) in enumerate(similar)
    ])[:12000]
    
    # Build prompt
    prompt = SUMMARY_PROMPT.format(
        target=target_doc[:24000],
        neighbors=neigh_text
    )
    
    # Generate summary
    return call_llm_endpoint(prompt)



## Generate Summary



In [ ]:

print(f"Processing complaint: {REFERENCE_NUMBER}")
print(f"Using model: Claude Sonnet 4.5 (Temperature: {LLM_TEMPERATURE})")
print(f"Retrieving {K_NEIGHBORS} similar complaints...")
print()

# Verify complaint exists
if REFERENCE_NUMBER not in doc_dict:
    print(f"ERROR: Complaint {REFERENCE_NUMBER} not found in pipeline.")
    print(f"Available reference numbers: {list(doc_dict.keys())[:10]}...")
    print()
    print("The complaint may not have been included in the pipeline setup.")
    print("Try running the 'RAG Pipeline Setup' notebook again to refresh the data.")
else:
    # Generate summary
    summary = generate_summary(REFERENCE_NUMBER, k_neighbors=K_NEIGHBORS)
    
    print("="*80)
    print(f"SUMMARY FOR COMPLAINT: {REFERENCE_NUMBER}")
    print("="*80)
    print()
    print(summary)
    print()
    print("="*80)



## View Similar Complaints



In [ ]:

# Get the similar complaints used for context
if REFERENCE_NUMBER in doc_dict:
    similar_complaints = retrieve_similar_chunks(REFERENCE_NUMBER, k=K_NEIGHBORS)
    
    print(f"Similar complaints used for context:\n")
    
    for i, (ref, score, doc) in enumerate(similar_complaints):
        print(f"{i+1}. {ref} (Similarity: {score:.3f})")
    
    print()
    print("These complaints were used to provide context and style guidance to the LLM.")



## Done

To generate a summary for a different complaint:
1. Change the reference number in the widget at the top
2. Run from the "Generate Summary" cell onwards (no need to reload pipeline)

If you get "complaint not found" errors:
- The complaint may not be in the pipeline
- Re-run the "RAG Pipeline Setup" notebook to refresh the data

